# Chemical-aware Wine Purchase Screening
All purchase records are synthetic. This notebook loads the measured checkpoint; it does not retrain. A flag means human review, not proof of fraud. Run from this project folder.

In [1]:
from pathlib import Path
import json
import pandas as pd
from demo import Screener
from experiment import ROOT, TX
screen = Screener()
wine = pd.read_csv(ROOT / "data/wine.csv")
purchases = pd.read_csv(ROOT / "data/synthetic_test.csv")
purchases.head()

  transaction_id  sample_id  ...  anomaly  scenario
0    SYN-102-7-0          7  ...        0    normal
1    SYN-102-7-1          7  ...        0    normal
2    SYN-102-7-2          7  ...        0    normal
3    SYN-102-7-3          7  ...        0    normal
4    SYN-102-7-4          7  ...        0    normal

[5 rows x 9 columns]

## Inspect one example from each scenario
Each purchase links to a laboratory sample. The chemical classifier compares its predicted class with the declared class. Grouped mean-occlusion scores indicate which feature groups support the screening score; correlated features and unrealistic substitutions limit this explanation.

In [2]:
examples = []
for scenario in purchases.scenario.unique():
    row = purchases[purchases.scenario == scenario].iloc[0]
    result = screen.predict(wine.drop(columns="target").iloc[int(row.sample_id)].to_numpy(), int(row.declared_cultivar), row[TX].to_numpy())
    examples.append({"scenario": scenario, **result})
examples

[{'scenario': 'normal', 'synthetic_demo': True, 'risk_score': 0.01670403778553009, 'threshold': 0.45000000000000007, 'action': 'no_model_flag', 'predicted_cultivar': 0, 'declared_cultivar': 0, 'chemical_disagreement_probability': 2.777576446533203e-05, 'top_factors': [{'factor': 'unit_price', 'risk_score_drop_at_training_mean': -0.00218903087079525}, {'factor': 'purchases_last_hour', 'risk_score_drop_at_training_mean': -0.0033364538103342056}, {'factor': 'new_merchant_category', 'risk_score_drop_at_training_mean': -0.00481843389570713}], 'diagnostic_factors': ['No simple diagnostic rule triggered; review the model score and supporting record.'], 'interpretation': 'Grouped mean-occlusion diagnostics, not causal explanations. Screening signal, not proof of fraud or chemical safety.'}, {'scenario': 'combined', 'synthetic_demo': True, 'risk_score': 0.9999908208847046, 'threshold': 0.45000000000000007, 'action': 'human_review', 'predicted_cultivar': 0, 'declared_cultivar': 2, 'chemical_disa

## Change a purchase and inspect its screening score
Change the declared cultivar or purchase fields below. Keep chemical feature order unchanged. These are simulation units, not verified market prices.

In [3]:
sample_id = int(purchases.iloc[0].sample_id)
declared_cultivar = 0
unit_price = 25
quantity = 6
purchases_last_hour = 1
new_merchant_category = 0
screen.predict(wine.drop(columns="target").iloc[sample_id].to_numpy(), declared_cultivar, [unit_price, quantity, purchases_last_hour, new_merchant_category])

{'synthetic_demo': True, 'risk_score': 0.023856038227677345, 'threshold': 0.45000000000000007, 'action': 'no_model_flag', 'predicted_cultivar': 0, 'declared_cultivar': 0, 'chemical_disagreement_probability': 2.777576446533203e-05, 'top_factors': [{'factor': 'unit_price', 'risk_score_drop_at_training_mean': 0.001328231766819954}, {'factor': 'purchases_last_hour', 'risk_score_drop_at_training_mean': -0.0045059919357299805}, {'factor': 'quantity', 'risk_score_drop_at_training_mean': -0.006158690899610519}], 'diagnostic_factors': ['No simple diagnostic rule triggered; review the model score and supporting record.'], 'interpretation': 'Grouped mean-occlusion diagnostics, not causal explanations. Screening signal, not proof of fraud or chemical safety.'}

In [4]:
metrics = json.loads((ROOT / "results/metrics.json").read_text())
pd.DataFrame([{ "model": c["name"], **{k:v for k,v in c["test"].items() if k != "confusion_matrix"}} for c in metrics["risk_candidates"]])

                 model  accuracy  ...     auroc    pr_auc
0     transaction_only  0.922222  ...  0.839250  0.760262
1      combined_linear  0.948148  ...  0.964659  0.926631
2         combined_mlp  0.940741  ...  0.971568  0.958092
3  combined_unweighted  0.940741  ...  0.971500  0.954492

[4 rows x 11 columns]

## Limits
There are only 27 independent chemical samples in the test partition. The transaction simulator deliberately includes label inconsistency; the measured gain is conditional on that assumption. Real deployment requires linked lab records, verified investigation outcomes, calibration, and recent external validation. The system cannot establish authenticity or food safety.